# Bangkok Environmental Data Warehouse — Step 1: Ingestion

Production-grade ingestion aligned with **Open-Meteo MCP** (get_weather_archive for historical weather, get_air_quality for AQ).  
Bronze (immutable .json.gz) → Silver (schema-enforced Parquet, year=YYYY/month=MM).  
Session UUID load_id, record_hash, ingestion_timestamp_utc; file logs; MD5 sidecars; checkpoint resume; solar-noon DQ audit.

In [1]:
# Cell 1: Install dependencies (run once in environment)
# %pip install -r requirements.txt
# Or: %pip install pandas>=2.0 pyarrow>=14 requests>=2.28

In [2]:
# Cell 2: Imports
from __future__ import annotations

import gc
import gzip
import hashlib
import json
import logging
import random
import time
import uuid
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests

In [3]:
# Cell 3: Config
PIPELINE_VERSION = "1.0.0"

# Paths (relative to project root)
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
BRONZE_WEATHER = DATA_ROOT / "bronze" / "openmeteo_weather"
BRONZE_AIRQUALITY = DATA_ROOT / "bronze" / "openmeteo_airquality"
SILVER_WEATHER = DATA_ROOT / "silver" / "openmeteo_weather"
SILVER_AIRQUALITY = DATA_ROOT / "silver" / "openmeteo_airquality"
CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints"
LOGS_DIR = PROJECT_ROOT / "logs"
METADATA_PATH = DATA_ROOT / "stations" / "bangkok_stations.parquet"

# Execution
CHUNK_MONTHS = 3
STATION_BATCH_SIZE = 20
BACKFILL_START = "2010-01-01T00:00:00Z"
INCREMENTAL_HOURS = 48
ARCHIVE_CUTOFF_MONTHS = 3  # Use weather_archive for data older than this

# Open-Meteo APIs (aligned with open-meteo MCP: get_weather_archive + get_air_quality)
OPENMETEO_ARCHIVE_BASE = "https://archive-api.open-meteo.com/v1/archive"
OPENMETEO_FORECAST_BASE = "https://api.open-meteo.com/v1/forecast"
OPENMETEO_AQ_BASE = "https://air-quality-api.open-meteo.com/v1/air-quality"
AIR4THAI_URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"

# API historical limits (skip and log 'Historical limit reached' if before these)
WEATHER_ARCHIVE_START = "1940-01-01"
AQ_HISTORICAL_START = "2013-01-01"  # AQ_MIN_DATE: allow ingestion when end_date >= this; skip 2010-2012

# Resilience (sequential calls only; no parallel API to avoid 429)
MAX_RETRIES = 5
BASE_BACKOFF_SEC = 2
JITTER_MIN, JITTER_MAX = 1, 3
CIRCUIT_BREAKER_SLEEP_SEC = 300
REQUEST_DELAY_SEC = 1.0  # Delay between API calls to avoid Open-Meteo rate limit (400/429)


In [4]:
# Cell 4: Logging setup — console by default; file logs/step1_<load_id>.log when load_id is set
def setup_logging(level: int = logging.INFO, load_id: str | None = None) -> logging.Logger:
    logger = logging.getLogger("bangkok_ingestion")
    logger.setLevel(level)
    if not logger.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
        logger.addHandler(h)
    if load_id:
        for h in list(logger.handlers):
            if isinstance(h, logging.FileHandler):
                logger.removeHandler(h)
        LOGS_DIR.mkdir(parents=True, exist_ok=True)
        log_path = LOGS_DIR / f"step1_run_{load_id}.log"
        fh = logging.FileHandler(log_path, encoding="utf-8")
        fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
        logger.addHandler(fh)
    return logger

LOG = setup_logging()

In [5]:
# Cell 5: Directory initialization
def ensure_dirs() -> None:
    for d in (BRONZE_WEATHER, BRONZE_AIRQUALITY, SILVER_WEATHER, SILVER_AIRQUALITY, CHECKPOINT_ROOT, LOGS_DIR, METADATA_PATH.parent):
        d.mkdir(parents=True, exist_ok=True)
    LOG.info("Directories initialized.")
ensure_dirs()

2026-02-22 12:41:52 [INFO] Directories initialized.


In [6]:
# Cell 6: Utility functions — time contract, hashing, MD5 checksum
def parse_timestamp_to_utc(value: Any) -> pd.Timestamp | None:
    """Normalize any timestamp to timezone-aware UTC. Returns None for invalid."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    try:
        if isinstance(value, (int, np.integer)):
            if value >= 1e12:
                ts = pd.Timestamp(value, unit="ms", tz="UTC")
            else:
                ts = pd.Timestamp(value, unit="s", tz="UTC")
        elif isinstance(value, str):
            ts = pd.to_datetime(value, utc=True)
        elif isinstance(value, pd.Timestamp):
            ts = value.tz_localize(None) if value.tz is None else value
            ts = ts.tz_convert("UTC")
        else:
            ts = pd.to_datetime(value, utc=True)
        return ts if ts.tz is not None else ts.tz_localize("UTC")
    except Exception:
        return None

def record_hash(row: pd.Series, columns: list[str]) -> str:
    key = "|".join(str(row.get(c, "")) for c in columns)
    return hashlib.sha256(key.encode()).hexdigest()[:16]

def now_utc() -> pd.Timestamp:
    return pd.Timestamp.now(tz="UTC")

def generate_checksum(file_path: Path) -> Path:
    """Compute MD5 of file and write sidecar <path>.md5. Returns path to .md5 file."""
    md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            md5.update(chunk)
    sidecar = file_path.with_suffix(file_path.suffix + ".md5")
    sidecar.write_text(md5.hexdigest() + "  " + file_path.name + "\n", encoding="utf-8")
    return sidecar

In [7]:
# Cell 7: Schema contracts (strict column order and dtypes)
WEATHER_SILVER_COLUMNS = [
    "stationID", "lat", "lon", "timestamp_utc", "timestamp_unix_ms",
    "temp_c", "humidity_pct", "pressure_hpa", "precipitation_mm", "wind_ms", "wind_dir_deg",
    "shortwave_radiation_wm2", "cloud_cover_pct", "u10_ms", "v10_ms",
    "data_source", "ingestion_timestamp_utc", "load_id", "pipeline_version", "record_hash"
]
WEATHER_SILVER_DTYPES = {
    "stationID": "string", "lat": "float64", "lon": "float64",
    "timestamp_utc": "datetime64[ns, UTC]", "timestamp_unix_ms": "int64",
    "temp_c": "float32", "humidity_pct": "float32", "pressure_hpa": "float32",
    "precipitation_mm": "float32", "wind_ms": "float32", "wind_dir_deg": "float32",
    "shortwave_radiation_wm2": "float32", "cloud_cover_pct": "float32",
    "u10_ms": "float32", "v10_ms": "float32",
    "data_source": "string", "ingestion_timestamp_utc": "datetime64[ns, UTC]",
    "load_id": "string", "pipeline_version": "string", "record_hash": "string"
}

AQ_SILVER_COLUMNS = [
    "stationID", "lat", "lon", "timestamp_utc", "timestamp_unix_ms",
    "pm2_5_ugm3", "pm10_ugm3", "no2_ugm3", "o3_ugm3", "so2_ugm3", "co_ugm3",
    "data_source", "ingestion_timestamp_utc", "load_id", "pipeline_version", "record_hash"
]
AQ_SILVER_DTYPES = {
    "stationID": "string", "lat": "float64", "lon": "float64",
    "timestamp_utc": "datetime64[ns, UTC]", "timestamp_unix_ms": "int64",
    "pm2_5_ugm3": "float32", "pm10_ugm3": "float32", "no2_ugm3": "float32",
    "o3_ugm3": "float32", "so2_ugm3": "float32", "co_ugm3": "float32",
    "data_source": "string", "ingestion_timestamp_utc": "datetime64[ns, UTC]",
    "load_id": "string", "pipeline_version": "string", "record_hash": "string"
}

In [8]:
# Cell 8: Deterministic 3-month chunks (no gaps, no overlaps, UTC)
def get_backfill_chunks() -> list[tuple[str, str]]:
    start = pd.Timestamp(BACKFILL_START, tz="UTC")
    end = now_utc()
    chunks = []
    s = start
    while s < end:
        e = s + pd.DateOffset(months=CHUNK_MONTHS) - pd.Timedelta(seconds=1)
        if e > end:
            e = end
        chunks.append((s.strftime("%Y-%m-%dT%H:%M:%SZ"), e.strftime("%Y-%m-%dT%H:%M:%SZ")))
        s = s + pd.DateOffset(months=CHUNK_MONTHS)
    return chunks

In [9]:
# Cell 9: Smart station discovery — case-insensitive substring 'bangkok' in areaEN; log and skip if missing
def fetch_air4thai_raw() -> list[dict]:
    resp = requests.get(AIR4THAI_URL, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("stations", data.get("data", []))

def extract_bangkok_stations() -> pd.DataFrame:
    raw = fetch_air4thai_raw()
    rows = []
    for r in raw:
        area = r.get("areaEN") or r.get("area_en") or r.get("AreaEN") or r.get("area")
        if area is None or (isinstance(area, str) and not area.strip()):
            LOG.info("Skipping record with missing/null areaEN: stationID=%s", r.get("stationID") or r.get("id"))
            continue
        area_str = str(area).strip()
        if "bangkok" not in area_str.lower():
            continue
        try:
            lat = float(r.get("lat", r.get("latitude", np.nan)))
            lon = float(r.get("lon", r.get("long", r.get("longitude", r.get("lng", np.nan)))))
        except (TypeError, ValueError):
            continue
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            continue
        station_id = r.get("stationID", r.get("station_id", r.get("id", "")))
        station_id = str(station_id).strip()
        if not station_id:
            continue
        name = r.get("nameEN") or r.get("nameTH") or r.get("stationName") or r.get("name") or station_id
        rows.append({
            "stationID": station_id,
            "stationName": str(name).strip() if name else station_id,
            "areaEN": area_str,
            "lat": lat,
            "lon": lon,
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["lat"] = df["lat"].astype("float64")
    df["lon"] = df["lon"].astype("float64")
    df["stationID"] = df["stationID"].astype("string")
    df["stationName"] = df["stationName"].astype("string")
    df["areaEN"] = df["areaEN"].astype("string")
    return df.drop_duplicates(subset=["stationID"]).reset_index(drop=True)

def print_bangkok_stations_table(stations: pd.DataFrame) -> None:
    if stations.empty:
        print("No Bangkok stations found.")
        return
    cols = ["stationID", "stationName", "areaEN"]
    display_df = stations[[c for c in cols if c in stations.columns]]
    print("Bangkok stations (ID, Name, Area):")
    print(display_df.to_string(index=False))

def save_station_metadata() -> pd.DataFrame:
    ensure_dirs()
    METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    stations = extract_bangkok_stations()
    print_bangkok_stations_table(stations)
    if stations.empty:
        LOG.warning("No Bangkok stations found from Air4Thai.")
        return stations
    stations.to_parquet(METADATA_PATH, index=False)
    LOG.info("Saved station metadata to data/stations: %s (%d stations)", METADATA_PATH.name, len(stations))
    return stations

In [10]:
# Cell 10: Load stations (from disk or fetch); print summary table first
def load_stations() -> pd.DataFrame:
    if METADATA_PATH.exists():
        stations = pd.read_parquet(METADATA_PATH)
        stations["stationID"] = stations["stationID"].astype("string")
        if "stationName" not in stations.columns:
            stations["stationName"] = stations["stationID"]
        print_bangkok_stations_table(stations)
        return stations
    return save_station_metadata()

In [11]:
# Cell 11: API wrapper — fallback when MCP unreachable: direct HTTP with exponential backoff (zero 429)
def _jitter() -> float:
    return random.uniform(JITTER_MIN, JITTER_MAX)

def _request_with_retry(url: str, params: dict[str, Any] | None = None, session: requests.Session | None = None, chunk_label: str | None = None, source: str = "API") -> dict | None:
    session = session or requests.Session()
    params = params or {}
    consecutive_5xx = 0
    for attempt in range(MAX_RETRIES):
        try:
            r = session.get(url, params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
            if 400 <= r.status_code < 500:
                try:
                    body = (r.text or "")[:400]
                    if body and chunk_label:
                        LOG.warning("[%s] HTTP %s for %s — skip chunk. Response: %s", chunk_label, r.status_code, source, body)
                    elif chunk_label:
                        LOG.warning("[%s] HTTP %s for %s — skip chunk", chunk_label, r.status_code, source)
                    else:
                        LOG.warning("HTTP %s (4xx) for %s — skip. Response: %s", r.status_code, url, body)
                except Exception:
                    if chunk_label:
                        LOG.warning("[%s] HTTP %s for %s — skip chunk", chunk_label, r.status_code, source)
                    else:
                        LOG.warning("HTTP %s (4xx) for %s — skip chunk", r.status_code, url)
                return None
            if r.status_code == 429:
                backoff = 60 + _jitter()
                LOG.warning("Rate limit 429; sleeping %.1fs", backoff)
                time.sleep(backoff)
                continue
            if 500 <= r.status_code < 600:
                consecutive_5xx += 1
                if consecutive_5xx >= MAX_RETRIES:
                    LOG.warning("Circuit breaker: 5 consecutive 5xx for %s", url)
                    time.sleep(CIRCUIT_BREAKER_SLEEP_SEC)
                    consecutive_5xx = 0
                    continue
            backoff = BASE_BACKOFF_SEC ** (attempt + 1) + _jitter()
            LOG.warning("HTTP %s for %s, retry in %.1fs", r.status_code, url, backoff)
            time.sleep(backoff)
        except Exception as e:
            LOG.warning("Request failed: %s", e)
            time.sleep(BASE_BACKOFF_SEC ** (attempt + 1) + _jitter())
    return None

In [12]:
# Cell 12: MCP tool mapping — get_weather_archive (old) / get_weather_forecast (recent); date validation
def _is_older_than_cutoff(end_str: str) -> bool:
    end_ts = pd.Timestamp(end_str[:10], tz="UTC")
    cutoff = now_utc() - pd.DateOffset(months=ARCHIVE_CUTOFF_MONTHS)
    return end_ts < cutoff

def _before_historical_limit(start_str: str, limit_date: str) -> bool:
    return start_str[:10] < limit_date

def fetch_weather_station(lat: float, lon: float, start: str, end: str, chunk_label: str | None = None) -> dict | None:
    if _before_historical_limit(start, WEATHER_ARCHIVE_START):
        LOG.info("Historical limit reached (weather): start %s < %s — skip", start[:10], WEATHER_ARCHIVE_START)
        return None
    hourly = "temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,shortwave_radiation,cloud_cover,wind_u_component_10m,wind_v_component_10m"
    if _is_older_than_cutoff(end):
        params = {
            "latitude": lat, "longitude": lon,
            "start_date": start[:10], "end_date": end[:10],
            "hourly": hourly,
            "timezone": "UTC"
        }
        return _request_with_retry(OPENMETEO_ARCHIVE_BASE, params, chunk_label=chunk_label, source="Weather")
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start[:10], "end_date": end[:10],
        "hourly": hourly,
        "timezone": "UTC"
    }
    return _request_with_retry(OPENMETEO_FORECAST_BASE, params, chunk_label=chunk_label, source="Weather")

In [13]:
# Cell 13: MCP get_air_quality — skip if end before AQ_HISTORICAL_START (2013-01-01)
def fetch_aq_station(lat: float, lon: float, start: str, end: str, chunk_label: str | None = None) -> dict | None:
    if end[:10] < AQ_HISTORICAL_START:
        LOG.info("Historical limit reached (AQ): end %s < %s — skip", end[:10], AQ_HISTORICAL_START)
        return None
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start[:10], "end_date": end[:10],
        "hourly": "pm2_5,pm10,nitrogen_dioxide,ozone,sulphur_dioxide,carbon_monoxide",
        "timezone": "UTC"
    }
    return _request_with_retry(OPENMETEO_AQ_BASE, params, chunk_label=chunk_label, source="Air Quality")

In [14]:
# Cell 14: Bronze writer — immutable JSON.gz under data/bronze/{weather|airquality}/YYYY/MM/
def write_bronze(payload: dict, layer: str, year: int, month: int, chunk_id: str, batch_id: str) -> Path:
    base = BRONZE_WEATHER if layer == "weather" else BRONZE_AIRQUALITY
    dir_path = base / str(year) / f"{month:02d}"
    dir_path.mkdir(parents=True, exist_ok=True)
    fname = f"batch_{chunk_id}_{batch_id}.json.gz"
    path = dir_path / fname
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    return path

In [15]:
# Cell 15: Silver transformer — raw JSON to schema-enforced DataFrame (weather)
def raw_weather_to_silver(
    raw: dict, station_id: str, lat: float, lon: float,
    load_id: str, ingestion_ts: pd.Timestamp
) -> pd.DataFrame | None:
    try:
        hourly = raw.get("hourly")
        if not hourly or "time" not in hourly:
            return None
        times = hourly["time"]
        n = len(times)
        df = pd.DataFrame({
            "stationID": [station_id] * n,
            "lat": [lat] * n, "lon": [lon] * n,
            "timestamp_utc": [parse_timestamp_to_utc(t) for t in times],
            "temp_c": hourly.get("temperature_2m", [None] * n),
            "humidity_pct": hourly.get("relative_humidity_2m", [None] * n),
            "pressure_hpa": hourly.get("surface_pressure", [None] * n),
            "precipitation_mm": hourly.get("precipitation", [None] * n),
            "wind_ms": hourly.get("wind_speed_10m", [None] * n),
            "wind_dir_deg": hourly.get("wind_direction_10m", [None] * n),
            "shortwave_radiation_wm2": hourly.get("shortwave_radiation", [None] * n),
            "cloud_cover_pct": hourly.get("cloud_cover", [None] * n),
        })
        # Convert wind U and V components from km/h to m/s
        wind_u_kmh = hourly.get("wind_u_component_10m", [None] * n)
        wind_v_kmh = hourly.get("wind_v_component_10m", [None] * n)
        df["u10_ms"] = [None if v is None else v / 3.6 for v in wind_u_kmh]
        df["v10_ms"] = [None if v is None else v / 3.6 for v in wind_v_kmh]
    except Exception as e:
        LOG.warning("raw_weather_to_silver failed: %s", e)
        return None
    df = df.dropna(subset=["timestamp_utc"])
    if df.empty:
        return None
    df["timestamp_unix_ms"] = df["timestamp_utc"].astype("int64") // 10**6
    df["data_source"] = "openmeteo_weather"
    df["ingestion_timestamp_utc"] = ingestion_ts
    df["load_id"] = load_id
    df["pipeline_version"] = PIPELINE_VERSION
    df["record_hash"] = df.apply(lambda r: record_hash(r, ["stationID", "timestamp_utc"]), axis=1)
    return enforce_weather_schema(df)

In [16]:
# Cell 16: Enforce weather schema (column order + dtypes); skip partition on mismatch
def enforce_weather_schema(df: pd.DataFrame) -> pd.DataFrame | None:
    for col in WEATHER_SILVER_COLUMNS:
        if col not in df.columns:
            LOG.error("Weather schema mismatch: missing column %s", col)
            return None
    df = df[WEATHER_SILVER_COLUMNS].copy()
    for col, dtype in WEATHER_SILVER_DTYPES.items():
        if col in ("timestamp_utc", "ingestion_timestamp_utc"):
            df[col] = pd.to_datetime(df[col], utc=True)
        else:
            df[col] = df[col].astype(dtype)
    return df

In [17]:
# Cell 17: Silver transformer — raw AQ JSON to schema-enforced DataFrame
def raw_aq_to_silver(
    raw: dict, station_id: str, lat: float, lon: float,
    load_id: str, ingestion_ts: pd.Timestamp
) -> pd.DataFrame | None:
    try:
        hourly = raw.get("hourly") or {}
        if "time" not in hourly:
            return None
        times = hourly["time"]
        n = len(times)
        df = pd.DataFrame({
            "stationID": [station_id] * n,
            "lat": [lat] * n, "lon": [lon] * n,
            "timestamp_utc": [parse_timestamp_to_utc(t) for t in times],
            "pm2_5_ugm3": hourly.get("pm2_5", [None] * n),
            "pm10_ugm3": hourly.get("pm10", [None] * n),
            "no2_ugm3": hourly.get("nitrogen_dioxide", hourly.get("no2", [None] * n)),
            "o3_ugm3": hourly.get("ozone", hourly.get("o3", [None] * n)),
            "so2_ugm3": hourly.get("sulphur_dioxide", hourly.get("so2", [None] * n)),
            "co_ugm3": hourly.get("carbon_monoxide", hourly.get("co", [None] * n)),
        })
    except Exception as e:
        LOG.warning("raw_aq_to_silver failed: %s", e)
        return None
    df = df.dropna(subset=["timestamp_utc"])
    if df.empty:
        return None
    df["timestamp_unix_ms"] = df["timestamp_utc"].astype("int64") // 10**6
    df["data_source"] = "openmeteo_airquality"
    df["ingestion_timestamp_utc"] = ingestion_ts
    df["load_id"] = load_id
    df["pipeline_version"] = PIPELINE_VERSION
    df["record_hash"] = df.apply(lambda r: record_hash(r, ["stationID", "timestamp_utc"]), axis=1)
    return enforce_aq_schema(df)

In [18]:
# Cell 18: Enforce AQ schema
def enforce_aq_schema(df: pd.DataFrame) -> pd.DataFrame | None:
    for col in AQ_SILVER_COLUMNS:
        if col not in df.columns:
            LOG.error("AQ schema mismatch: missing column %s", col)
            return None
    df = df[AQ_SILVER_COLUMNS].copy()
    for col, dtype in AQ_SILVER_DTYPES.items():
        if col in ("timestamp_utc", "ingestion_timestamp_utc"):
            df[col] = pd.to_datetime(df[col], utc=True)
        else:
            df[col] = df[col].astype(dtype)
    return df

In [19]:
# Cell 19: Silver writer — atomic write (tmp then rename), partition year=YYYY/month=MM
def write_silver_partition(df: pd.DataFrame, layer: str, partition_year: int, partition_month: int) -> Path | None:
    base = SILVER_WEATHER if layer == "weather" else SILVER_AIRQUALITY
    part_dir = base / f"year={partition_year}" / f"month={partition_month:02d}"
    part_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir = base / "tmp"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    safe_id = f"{partition_year}{partition_month:02d}_{id(df)}_{int(pd.Timestamp.now().value)}"
    tmp_path = tmp_dir / f"part_{safe_id}.parquet"
    try:
        df.to_parquet(tmp_path, index=False)
        final_path = part_dir / f"part_{safe_id}.parquet"
        tmp_path.rename(final_path)
        generate_checksum(final_path)
        return final_path
    except Exception as e:
        LOG.error("Silver write failed: %s", e)
        if tmp_path.exists():
            tmp_path.unlink(missing_ok=True)
        return None

In [20]:
# Cell 20: Checkpoint handler — checkpoints/year=YYYY/chunk=<chunk_id>/batch=<batch_id>.done
def checkpoint_path(chunk_id: str, batch_id: str, year: int) -> Path:
    d = CHECKPOINT_ROOT / f"year={year}" / f"chunk={chunk_id}" / f"batch={batch_id}.done"
    return d

def checkpoint_exists(chunk_id: str, batch_id: str, year: int) -> bool:
    return checkpoint_path(chunk_id, batch_id, year).exists()

def write_checkpoint(chunk_id: str, batch_id: str, year: int, no_data: bool = False) -> None:
    p = checkpoint_path(chunk_id, batch_id, year)
    p.parent.mkdir(parents=True, exist_ok=True)
    lines = [f"done\t{now_utc().isoformat()}\n"]
    if no_data:
        lines.append("no_data\tTrue\n")
    p.write_text("".join(lines))

In [21]:
# Cell 21: Data quality (per-chunk) — missing %, pm2_5 bounds, radiation at night, min/max timestamp
def audit_chunk_quality(weather_dfs: list[pd.DataFrame], aq_dfs: list[pd.DataFrame], chunk_label: str) -> None:
    if not aq_dfs and not weather_dfs:
        return
    for name, dfs in [("weather", weather_dfs), ("aq", aq_dfs)]:
        if not dfs:
            continue
        combined = pd.concat(dfs, ignore_index=True)
        if combined.empty:
            continue
        if "timestamp_utc" in combined.columns:
            ts = pd.to_datetime(combined["timestamp_utc"], utc=True)
            LOG.info("[DQ %s] %s min_ts=%s max_ts=%s", chunk_label, name, ts.min(), ts.max())
        if name == "aq" and "pm2_5_ugm3" in combined.columns:
            pm = combined["pm2_5_ugm3"].dropna()
            neg = (pm < 0).sum()
            over = (pm > 1000).sum()
            missing_pct = (1 - combined["pm2_5_ugm3"].notna().mean()) * 100
            LOG.info("[DQ aq] %s missing_pct=%.2f pm2_5<0=%s pm2_5>1000=%s", chunk_label, missing_pct, int(neg), int(over))
        if name == "weather" and "shortwave_radiation_wm2" in combined.columns and "timestamp_utc" in combined.columns:
            ts = pd.to_datetime(combined["timestamp_utc"], utc=True)
            hour = ts.dt.hour
            night = (hour < 6) | (hour >= 18)
            rad_night = combined.loc[night, "shortwave_radiation_wm2"]
            anomaly = (rad_night > 50).sum() if len(rad_night) else 0
            LOG.info("[DQ weather] %s radiation_at_night_anomaly_count=%s", chunk_label, int(anomaly))
        del combined
    gc.collect()

In [22]:
# Cell 22: Backfill runner — session UUID; 3-month chunks; chunk_label for observability; checkpoint skip (idempotent)
def run_backfill() -> None:
    ensure_dirs()
    load_id = str(uuid.uuid4())
    setup_logging(load_id=load_id)
    ingestion_ts = now_utc()
    stations = load_stations()
    if stations.empty:
        LOG.error("No stations; run save_station_metadata() first.")
        return
    stations["stationID"] = stations["stationID"].astype("string")
    chunks = get_backfill_chunks()
    LOG.info("Backfill load_id=%s chunks=%d stations=%d", load_id, len(chunks), len(stations))

    for chunk_idx, (start, end) in enumerate(chunks):
        chunk_id = start[:10].replace("-", "")
        chunk_label = pd.to_datetime(start).strftime("%b %Y")
        year_start = int(start[:4])
        LOG.info("--- [%s] Processing Chunk %s ---", chunk_label, chunk_id)

        for batch_start in range(0, len(stations), STATION_BATCH_SIZE):
            batch = stations.iloc[batch_start : batch_start + STATION_BATCH_SIZE]
            batch_id = f"{batch_start:04d}"
            if checkpoint_exists(chunk_id, batch_id, year_start):
                LOG.info("Skip chunk=%s batch=%s (checkpoint exists)", chunk_id, batch_id)
                continue

            batch_had_data = False
            for _, row in batch.iterrows():
                sid, lat, lon = str(row["stationID"]), float(row["lat"]), float(row["lon"])
                raw_w = fetch_weather_station(lat, lon, start, end, chunk_label=chunk_label)
                if REQUEST_DELAY_SEC > 0:
                    time.sleep(REQUEST_DELAY_SEC)
                if raw_w:
                    batch_had_data = True
                    write_bronze(raw_w, "weather", year_start, int(start[5:7]), chunk_id, batch_id)
                    df_w = raw_weather_to_silver(raw_w, sid, lat, lon, load_id, ingestion_ts)
                    if df_w is not None and not df_w.empty:
                        for (py, pm), g in df_w.groupby([df_w["timestamp_utc"].dt.year, df_w["timestamp_utc"].dt.month]):
                            write_silver_partition(g, "weather", int(py), int(pm))
                    del df_w
                del raw_w
                gc.collect()

                raw_aq = fetch_aq_station(lat, lon, start, end, chunk_label=chunk_label)
                if REQUEST_DELAY_SEC > 0:
                    time.sleep(REQUEST_DELAY_SEC)
                if raw_aq:
                    batch_had_data = True
                    write_bronze(raw_aq, "airquality", year_start, int(start[5:7]), chunk_id, batch_id)
                    df_aq = raw_aq_to_silver(raw_aq, sid, lat, lon, load_id, ingestion_ts)
                    if df_aq is not None and not df_aq.empty:
                        for (py, pm), g in df_aq.groupby([df_aq["timestamp_utc"].dt.year, df_aq["timestamp_utc"].dt.month]):
                            write_silver_partition(g, "airquality", int(py), int(pm))
                    del df_aq
                del raw_aq
                gc.collect()

            write_checkpoint(chunk_id, batch_id, year_start, no_data=not batch_had_data)
            del batch
            gc.collect()

    LOG.info("Backfill completed load_id=%s", load_id)

In [23]:
# Cell 23: Incremental runner — last 48 hours; merge into year=YYYY/month=MM; dedupe (stationID, timestamp_utc)
def run_incremental() -> None:
    ensure_dirs()
    load_id = now_utc().strftime("%Y%m%d%H%M%S") + "_inc_" + str(random.randint(1000, 9999))
    ingestion_ts = now_utc()
    end_utc = now_utc()
    start_utc = end_utc - pd.Timedelta(hours=INCREMENTAL_HOURS)
    start_str = start_utc.strftime("%Y-%m-%dT%H:%M:%SZ")
    end_str = end_utc.strftime("%Y-%m-%dT%H:%M:%SZ")

    stations = load_stations()
    if stations.empty:
        LOG.error("No stations.")
        return
    stations["stationID"] = stations["stationID"].astype("string")

    for _, row in stations.iterrows():
        sid, lat, lon = str(row["stationID"]), float(row["lat"]), float(row["lon"])
        raw_w = fetch_weather_station(lat, lon, start_str, end_str)
        if raw_w:
            year_start, month_start = start_utc.year, start_utc.month
            write_bronze(raw_w, "weather", year_start, month_start, "inc", sid[:32])
            df_w = raw_weather_to_silver(raw_w, sid, lat, lon, load_id, ingestion_ts)
            if df_w is not None and not df_w.empty:
                for (py, pm), g in df_w.groupby([df_w["timestamp_utc"].dt.year, df_w["timestamp_utc"].dt.month]):
                    _merge_into_silver(g, "weather", int(py), int(pm))
            del df_w
        del raw_w
        gc.collect()

        raw_aq = fetch_aq_station(lat, lon, start_str, end_str)
        if raw_aq:
            write_bronze(raw_aq, "airquality", year_start, month_start, "inc", sid[:32])
            df_aq = raw_aq_to_silver(raw_aq, sid, lat, lon, load_id, ingestion_ts)
            if df_aq is not None and not df_aq.empty:
                for (py, pm), g in df_aq.groupby([df_aq["timestamp_utc"].dt.year, df_aq["timestamp_utc"].dt.month]):
                    _merge_into_silver(g, "airquality", int(py), int(pm))
            del df_aq
        del raw_aq
        gc.collect()

    LOG.info("Incremental completed load_id=%s", load_id)

In [24]:
# Cell 24: Merge incremental into existing partition; deduplicate (stationID, timestamp_utc)
def _merge_into_silver(new_df: pd.DataFrame, layer: str, partition_year: int, partition_month: int) -> None:
    base = SILVER_WEATHER if layer == "weather" else SILVER_AIRQUALITY
    part_dir = base / f"year={partition_year}" / f"month={partition_month:02d}"
    part_dir.mkdir(parents=True, exist_ok=True)
    existing = list(part_dir.glob("*.parquet"))
    if not existing:
        write_silver_partition(new_df, layer, partition_year, partition_month)
        return
    existing_df = pd.concat([pd.read_parquet(p) for p in existing], ignore_index=True)
    combined = pd.concat([existing_df, new_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=["stationID", "timestamp_utc"], keep="last")
    cols = WEATHER_SILVER_COLUMNS if layer == "weather" else AQ_SILVER_COLUMNS
    combined = combined[[c for c in cols if c in combined.columns]]
    if set(combined.columns) != set(cols):
        LOG.error("Schema mismatch on merge; skipping partition")
        return
    combined = combined[cols]
    tmp_dir = base / "tmp"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    safe_id = f"merged_{partition_year}{partition_month:02d}_{int(pd.Timestamp.now().value)}"
    tmp_path = tmp_dir / f"{safe_id}.parquet"
    combined.to_parquet(tmp_path, index=False)
    for p in existing:
        p.unlink()
    final_path = part_dir / f"{safe_id}.parquet"
    tmp_path.rename(final_path)
    generate_checksum(final_path)
    del existing_df, combined
    gc.collect()

In [25]:
# Cell 25: Data quality audit — Silver coverage; solar-noon audit (shortwave ~0 during 18:00–06:00 UTC+7)
# Night UTC+7 = 18:00–06:00 local = 11:00 UTC to 05:00 UTC (next day) → UTC hour in [0,1,2,3,4,5] or [11..23]
NIGHT_UTC_HOURS = set(range(0, 6)) | set(range(11, 24))
RAD_NIGHT_THRESHOLD_WM2 = 20  # shortwave_radiation_wm2 above this at night = anomaly

def run_dq_audit(silver_base: Path | None = None, year: int | None = None, month: int | None = None) -> None:
    if silver_base is None:
        run_dq_audit(SILVER_WEATHER, year, month)
        run_dq_audit(SILVER_AIRQUALITY, year, month)
        return
    base = silver_base
    if year is not None and month is not None:
        part_dir = base / f"year={year}" / f"month={month:02d}"
        paths = list(part_dir.glob("*.parquet")) if part_dir.exists() else []
    else:
        paths = list(base.rglob("*.parquet"))
    if not paths:
        LOG.warning("No Silver Parquet found under %s", base)
        return
    df = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
    if "timestamp_utc" in df.columns:
        ts = pd.to_datetime(df["timestamp_utc"], utc=True)
        LOG.info("[DQ] min_ts=%s max_ts=%s rows=%d", ts.min(), ts.max(), len(df))
    if "pm2_5_ugm3" in df.columns:
        pm = df["pm2_5_ugm3"].dropna()
        missing_pct = (1 - df["pm2_5_ugm3"].notna().mean()) * 100
        LOG.info("[DQ AQ] missing_pct=%.2f pm2_5<0=%s pm2_5>1000=%s", missing_pct, (pm < 0).sum(), (pm > 1000).sum())
    if "shortwave_radiation_wm2" in df.columns and "timestamp_utc" in df.columns:
        ts = pd.to_datetime(df["timestamp_utc"], utc=True)
        hour_utc = ts.dt.hour
        night_mask = hour_utc.isin(NIGHT_UTC_HOURS)
        rad_night = df.loc[night_mask, "shortwave_radiation_wm2"].dropna()
        anomalies = (rad_night > RAD_NIGHT_THRESHOLD_WM2).sum() if len(rad_night) else 0
        LOG.info("[DQ Weather] solar-noon audit: shortwave_radiation_wm2 ~0 during 18:00–06:00 UTC+7; anomaly_count=%s", int(anomalies))
    del df
    gc.collect()

## Run instructions

- **Run All**: Execute all cells. The final cell runs `save_station_metadata()`, `run_backfill()`, then `run_dq_audit()`.
- **Logs**: Each backfill writes to `logs/step1_run_<load_id>.log` (load_id = UUID).
- **Checkpoints**: Completed station batches are skipped on resume; 16-year backfill is resumable after any crash.
- **ถ้า Silver หยุดที่เดือนใดเดือนหนึ่ง (เช่น 2017/09)**: มักเป็นเพราะ Open-Meteo ส่ง HTTP 400 (rate limit หรือ quota). Log จะแสดงข้อความจาก API ในบรรทัด "Response: ...". ใช้ `REQUEST_DELAY_SEC = 1` ใน Config เพื่อหน่วงระหว่าง request; ถ้าต้องการดึงช่วงที่ขาด ให้ลบ checkpoint ตั้งแต่ chunk นั้นเป็นต้นไปแล้วรันใหม่.
- **Incremental**: After backfill, run Cell 23 `run_incremental()` for last 48 hours; merge + dedupe by `(stationID, timestamp_utc)`.
- **Integrity**: Every Silver Parquet has a sidecar `.parquet.md5`; stationID is always string; lineage: load_id, record_hash, ingestion_timestamp_utc.

In [26]:

   
   # Cell 27: Full execution — Run All triggers: stations → backfill → DQ audit
save_station_metadata()
run_backfill()
run_dq_audit()

2026-02-22 12:41:53 [INFO] Directories initialized.
2026-02-22 12:41:53 [INFO] Saved station metadata to data/stations: bangkok_stations.parquet (79 stations)
2026-02-22 12:41:53 [INFO] Directories initialized.
2026-02-22 12:41:53 [INFO] Backfill load_id=0f54f52c-e8f4-446e-8f73-bcde0efb1991 chunks=65 stations=79
2026-02-22 12:41:53 [INFO] --- [Jan 2010] Processing Chunk 20100101 ---
2026-02-22 12:41:53 [INFO] Skip chunk=20100101 batch=0000 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20100101 batch=0020 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20100101 batch=0040 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20100101 batch=0060 (checkpoint exists)
2026-02-22 12:41:53 [INFO] --- [Apr 2010] Processing Chunk 20100401 ---
2026-02-22 12:41:53 [INFO] Skip chunk=20100401 batch=0000 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20100401 batch=0020 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20100401 batch=0040 (checkpoint exis

Bangkok stations (ID, Name, Area):
stationID                                                                      stationName                                                                                           areaEN
      02t                                           Bansomdejchaopraya Rajabhat University                                                             Hiran Ruchi, Khet Thon Buri, Bangkok
      03t                                                       Highway NO.3902 km.13 +600                                                      Kanchanaphisek Rd, Bang Khun Thian, Bangkok
      05t                                                   Thai Meteorological Department                                                                   Bang Na, Khet Bang Na, Bangkok
      12t                                                            Nonsi Witthaya School                                                               Chong Nonsi, Khet Yannawa, Bangkok
      50t                

2026-02-22 12:41:53 [INFO] Skip chunk=20110701 batch=0020 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20110701 batch=0040 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20110701 batch=0060 (checkpoint exists)
2026-02-22 12:41:53 [INFO] --- [Oct 2011] Processing Chunk 20111001 ---
2026-02-22 12:41:53 [INFO] Skip chunk=20111001 batch=0000 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20111001 batch=0020 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20111001 batch=0040 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20111001 batch=0060 (checkpoint exists)
2026-02-22 12:41:53 [INFO] --- [Jan 2012] Processing Chunk 20120101 ---
2026-02-22 12:41:53 [INFO] Skip chunk=20120101 batch=0000 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20120101 batch=0020 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20120101 batch=0040 (checkpoint exists)
2026-02-22 12:41:53 [INFO] Skip chunk=20120101 batch=0060 (checkpoint exists

KeyboardInterrupt: 